In [33]:
import os, re, hashlib
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
N_SPLITS = 20  
MANIFEST = "/kaggle/input/datasets/allumalla07/plantvillage-final-deduplicated/plantvillage_final_deduplicated.csv"
DATASET_ROOT = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
OUT = "/kaggle/working/split.csv"

In [34]:
df = pd.read_csv(MANIFEST)
df["class"] = df["label"]
df["abs_path"] = df["relative_path"].apply(lambda p: os.path.join(DATASET_ROOT, p))

found = df["abs_path"].apply(os.path.isfile)
print("Images found:", found.sum(), "/", len(df))
assert found.all()

Images found: 53705 / 53705


In [35]:
def group_key(row):
    fn = row["relative_path"].split("/")[-1].split("___", 1)[-1]
    fn = re.sub(r"\s+copy(\s+\d+)?(?=\.[^.]+$)", "", fn, flags=re.I)
    return f'{row["class"]}|{fn.lower()}'

df["group"] = df.apply(group_key, axis=1)
sizes = df.groupby("group").size()
print("Groups:", len(sizes), "| groups with >1 image:", (sizes > 1).sum())

Groups: 52939 | groups with >1 image: 603


In [36]:
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
df["fold"] = -1
for fold, (_, idx) in enumerate(sgkf.split(df, y=df["class"], groups=df["group"])):
    df.iloc[idx, df.columns.get_loc("fold")] = fold

assert (df["fold"] >= 0).all()
df["split"] = df["fold"].map(lambda f: "test" if f < 3 else "val" if f < 6 else "train")

print("Before moving duplicates to train:")
print(df["split"].value_counts())

Before moving duplicates to train:
split
train    37602
test      8056
val       8047
Name: count, dtype: int64


In [39]:
df["group_size"] = df.groupby("group")["group"].transform("size")
multi = df["group_size"] > 1
df.loc[multi, "split"] = "train"

print("After moving duplicates to train:")
print(df["split"].value_counts())
print((df["split"].value_counts(normalize=True) * 100).round(2))

After moving duplicates to train:
split
train    37987
val       7862
test      7856
Name: count, dtype: int64
split
train    70.73
val      14.64
test     14.63
Name: proportion, dtype: float64


In [40]:
import numpy as np
print("5b shared-pool version running")

rng = np.random.RandomState(SEED)
n_per_class = df.groupby("class").size()

for cls, n in n_per_class.items():
    target = round(n * 0.15)
    cur = {sp: ((df["class"] == cls) & (df["split"] == sp)).sum() for sp in ["val", "test"]}
    need = {sp: max(target - cur[sp], 0) for sp in cur}
    total_need = need["val"] + need["test"]
    if total_need == 0:
        continue

    pool = df.index[(df["class"] == cls) & (df["split"] == "train") & (df["group_size"] == 1)].to_numpy()
    pool = rng.permutation(pool)
    take = min(total_need, len(pool))


    n_val = int(round(take * need["val"] / total_need))
    df.loc[pool[:n_val], "split"] = "val"
    df.loc[pool[n_val:take], "split"] = "test"

print("After rebalancing:")
print(df["split"].value_counts())
print((df["split"].value_counts(normalize=True) * 100).round(2))
display(pd.crosstab(df["class"], df["split"]).loc[["Corn_(maize)___healthy",
                                                    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
                                                    "Potato___healthy"]])

5b shared-pool version running
After rebalancing:
split
train    37751
test      7978
val       7976
Name: count, dtype: int64
split
train    70.29
test     14.86
val      14.85
Name: proportion, dtype: float64


split,test,train,val
class,,,
Corn_(maize)___healthy,68,1025,66
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,77,359,77
Potato___healthy,23,105,24


In [41]:
from concurrent.futures import ThreadPoolExecutor

dup_paths = df["relative_path"].duplicated().sum()
bad_groups = (df.groupby("group")["split"].nunique() > 1).sum()

df["size"] = df["abs_path"].apply(os.path.getsize)
needs_hash = df["size"].duplicated(keep=False)
print("Files needing MD5:", needs_hash.sum(), "/", len(df))

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

df["md5"] = df["abs_path"] 
paths_to_hash = df.loc[needs_hash, "abs_path"].tolist()
with ThreadPoolExecutor(max_workers=16) as ex:
    hashes = list(ex.map(md5, paths_to_hash))
df.loc[needs_hash, "md5"] = hashes

bad_md5 = (df.groupby("md5")["split"].nunique() > 1).sum()
cls_per_split = df.groupby("split")["class"].nunique()


dups_in_eval = ((df["group_size"] > 1) & (df["split"] != "train")).sum()

print("Duplicate paths:", dup_paths)
print("Groups spanning >1 split:", bad_groups)
print("Identical files across splits:", bad_md5)
print("Near-duplicate images in val/test:", dups_in_eval)
print("Classes per split:\n", cls_per_split)
display(pd.crosstab(df["class"], df["split"]))

assert dup_paths == 0
assert bad_groups == 0
assert bad_md5 == 0
assert dups_in_eval == 0
assert (cls_per_split == df["class"].nunique()).all()
print("\nAll checks passed.")

Files needing MD5: 49680 / 53705
Duplicate paths: 0
Groups spanning >1 split: 0
Identical files across splits: 0
Near-duplicate images in val/test: 0
Classes per split:
 split
test     38
train    38
val      38
Name: class, dtype: int64


split,test,train,val
class,,,
Apple___Apple_scab,96,440,94
Apple___Black_rot,93,430,95
Apple___Cedar_apple_rust,42,192,41
Apple___healthy,249,1143,246
Blueberry___healthy,224,1048,224
Cherry_(including_sour)___Powdery_mildew,156,726,157
Cherry_(including_sour)___healthy,126,586,126
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,77,359,77
Corn_(maize)___Common_rust_,179,833,180



All checks passed.


In [42]:
split_df = df[["relative_path", "class", "split"]].rename(columns={"relative_path": "path"})
split_df = split_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split_df.to_csv(OUT, index=False)

check = pd.read_csv(OUT)
print(check.columns.tolist(), len(check))
display(check.head())

['path', 'class', 'split'] 53705


,path,class,split
0,Soybean___healthy/85478cac-7aed-4797-982b-410a...,Soybean___healthy,train
1,Tomato___Tomato_Yellow_Leaf_Curl_Virus/1a867f1...,Tomato___Tomato_Yellow_Leaf_Curl_Virus,train
2,Tomato___Leaf_Mold/30f6aead-5cd9-453d-85e6-56e...,Tomato___Leaf_Mold,train
3,Apple___healthy/9decdcd8-8027-4901-8da6-bd27c0...,Apple___healthy,train
4,Peach___Bacterial_spot/b269e00a-55a1-4e18-bab5...,Peach___Bacterial_spot,train
